# Evaluate DeNICE task 5, round 19 on Kaggle

Attach the `100-clients` data directory and the `1208-denice` checkpoint folder before running. The checkpoint folder must contain `checkpoint_task_5_base.pt` plus `checkpoint_task_5_round_0.pt` through `checkpoint_task_5_round_19.pt`. Enable Internet so the notebook can clone the matching evaluator code. Evaluation assigns every test sample only to a client whose saved router covers that sample's true task episode; unsupported samples are reported explicitly.


In [ ]:
# Cell 1 — clone evaluator code
from pathlib import Path
import shutil
import subprocess
import sys

REPO_PATH = Path('/kaggle/working/FL_IL_IDS')
if REPO_PATH.exists():
    shutil.rmtree(REPO_PATH)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/khoilv2005/FL_IL_IDS.git', str(REPO_PATH),
], check=True)
print('Repository:', subprocess.check_output(['git', '-C', str(REPO_PATH), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Cell 2 — configure Kaggle Input paths
# Data and extracted checkpoint-folder paths attached to this Kaggle notebook.
KAGGLE_INPUT = Path('/kaggle/input')
DATA_DIR = Path('/kaggle/input/datasets/khoilv2005/100-clients/100-clients')
CHECKPOINT_DIR = Path('/kaggle/input/datasets/khoilv2005/1208-denice')
CHECKPOINT_NAME = 'checkpoint_task_5_round_19.pt'

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_DIR}. Edit Cell 2.')

if not CHECKPOINT_DIR.is_dir():
    raise FileNotFoundError(f'Checkpoint folder does not exist: {CHECKPOINT_DIR}')
print('Dataset:', DATA_DIR)
print('Checkpoint folder:', CHECKPOINT_DIR)


In [ ]:
# Cell 3 — find the final checkpoint in the attached Kaggle Input folder
checkpoint_candidates = sorted(CHECKPOINT_DIR.rglob(CHECKPOINT_NAME))
if len(checkpoint_candidates) != 1:
    raise RuntimeError(f'Expected exactly one final checkpoint; found: {checkpoint_candidates}')
CHECKPOINT_PATH = checkpoint_candidates[0]
print('Checkpoint selected:', CHECKPOINT_PATH)


In [ ]:
# Cell 4 — validate task 5 / round 19 and every required delta dependency
import torch

header = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert header.get('algorithm') == 'denice', header.get('algorithm')
assert int(header.get('task_id', header.get('task', -1))) == 5, header.get('task_id')
assert int(header.get('round_id', header.get('final_round_id', -1))) == 19, header.get('round_id')

if header.get('checkpoint_type') == 'denice_delta_round':
    required = [CHECKPOINT_PATH.parent / header['base_path']]
    cursor = header
    while cursor.get('previous_round_path'):
        previous = CHECKPOINT_PATH.parent / cursor['previous_round_path']
        required.append(previous)
        if not previous.is_file():
            raise FileNotFoundError(f'Missing required delta: {previous}')
        cursor = torch.load(previous, map_location='cpu', weights_only=False)
    print(f'Delta chain verified: {len(required)} dependency files.')
else:
    print('Full task-end checkpoint verified.')


In [ ]:
# Cell 5 — coverage-aware cumulative evaluation; no client receives an unsupported task episode
EVAL_OUTPUT = Path('/kaggle/working/denice_task5_round19_eval.json')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
command = [
    sys.executable, str(REPO_PATH / 'eval_checkpoint.py'),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--data-dir', str(DATA_DIR),
    '--device', device,
    '--router-mode', 'multiclass',
    '--evaluation-mode', 'coverage_aware_local',
    # topk=1 is identical to hard, so only run distinct modes.
    '--route-modes', 'hard,nomask',
    '--eval-seed', '42',
    '--output', str(EVAL_OUTPUT),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
print(f'✓ Evaluation saved to Kaggle Output: {EVAL_OUTPUT}')


In [ ]:
# Cell 6 — show saved result
import json
with open(EVAL_OUTPUT, 'r', encoding='utf-8') as f:
    evaluation = json.load(f)
print(json.dumps(evaluation, indent=2))
